In [1]:
import sys
import json
import numpy as np
import torch
import cv2
import trimesh
from pathlib import Path
from scipy.spatial.transform import Rotation
from PIL import Image
import nvdiffrast.torch as dr

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # use the less-busy GPU

In [ ]:
PROJECT_ROOT = Path(".").resolve().parent

FP_ROOT = Path.home() / "rai_workspace/FoundationPose"
sys.path.insert(0, str(FP_ROOT))
sys.path.insert(0, str(FP_ROOT / "mycuda"))

GD_CONFIG  = Path.home() / "rai_workspace/grounding_dino_weights/GroundingDINO_SwinT_OGC.py"
GD_WEIGHTS = Path.home() / "rai_workspace/grounding_dino_weights/groundingdino_swint_ogc.pth"
MESH_PATH  = PROJECT_ROOT / "megapose_objects/lamp/mesh.ply"
CAM_INFO   = PROJECT_ROOT / "outputs/camera_info.json"

DETECTION_PROMPT    = "yellow lamp"
SAM3_SCORE_THRESH   = 0.10    # minimum SAM3 instance score to keep
SAM3_MASK_THRESH    = 0.50    # SAM3 mask binarisation threshold
GD_BOX_THRESHOLD    = 0.25    # GroundingDINO box confidence (fallback only)
GD_TEXT_THRESHOLD   = 0.20
MAX_PER_CAMERA      = 2       # keep up to 2 detections per camera
MAX_TOTAL           = 8       # hard cap on FoundationPose runs
CLUSTER_DIST_M      = 0.30    # world-frame radius for merging duplicate estimates
N_FP_ITERATIONS     = 5       # FoundationPose register() refinement iterations
DEVICE              = "cuda"

# Ground-truth poses [x, y, z, qw, qx, qy, qz] from mesh_clutter_high_30.g
GT = {
    "goal_yellow_lamp_9_mesh":       np.array([-0.620966, -0.342511,  0.671156,
                                                0.287543, -0.46152,   0.714029,  0.441]),
    "goal_pose_yellow_lamp_12_mesh": np.array([ 0.219547,  0.280159,  0.698313,
                                                0.38692,   0.0,       0.0,      -0.922113]),
}

In [3]:
def boxes_cxcywh_to_xyxy(boxes_norm: torch.Tensor, W: int, H: int) -> torch.Tensor:
    cx, cy, w, h = boxes_norm.unbind(-1)
    return torch.stack([(cx - w/2)*W, (cy - h/2)*H,
                        (cx + w/2)*W, (cy + h/2)*H], dim=-1)


def project_pt(pt3d, K):
    p = K @ np.asarray(pt3d, dtype=float)
    return (int(p[0]/p[2]), int(p[1]/p[2]))

In [4]:
print("Loading SAM3 …")
from transformers import Sam3Processor, Sam3Model

sam3_processor = Sam3Processor.from_pretrained("facebook/sam3")
sam3_model     = Sam3Model.from_pretrained("facebook/sam3").to(DEVICE)
sam3_model.eval()

# GroundingDINO loaded lazily — only if SAM3 text fails on a camera
gd_model = None

def load_gdino():
    global gd_model
    if gd_model is None:
        print("  [fallback] Loading GroundingDINO …")
        from groundingdino.util.inference import load_model
        gd_model = load_model(str(GD_CONFIG), str(GD_WEIGHTS))
        gd_model.eval()
    return gd_model

Loading SAM3 …


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

In [5]:
print("Loading FoundationPose …")
from estimater import FoundationPose, PoseRefinePredictor, ScorePredictor

scorer  = ScorePredictor()
refiner = PoseRefinePredictor()
glctx   = dr.RasterizeCudaContext()

mesh = trimesh.load(str(MESH_PATH))
fp_estimator = FoundationPose(
    model_pts     = np.array(mesh.vertices,       dtype=np.float32),
    model_normals = np.array(mesh.vertex_normals,  dtype=np.float32),
    mesh          = mesh,
    scorer        = scorer,
    refiner       = refiner,
    debug_dir     = "/tmp/fp_debug",
    debug         = 0,
    glctx         = glctx,
)

Loading FoundationPose …
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


[__init__()] self.cfg: 
 lr: 0.0001
c_in: 6
zfar: 'Infinity'
debug: null
n_view: 1
run_id: 3wy8qqex
use_BN: true
exp_name: 2024-01-11-20-02-45
n_epochs: 62
save_dir: /home/bowenw/debug/2024-01-11-20-02-45/
use_mask: false
loss_type: pairwise_valid
optimizer: adam
batch_size: 64
crop_ratio: 1.1
enable_amp: true
use_normal: false
max_num_key: null
warmup_step: -1
input_resize:
- 160
- 160
max_step_val: 1000
vis_interval: 1000
weight_decay: 0
normalize_xyz: true
resume_run_id: null
clip_grad_norm: 'Infinity'
lr_epoch_decay: 500
render_backend: nvdiffrast
train_num_pair: 5
lr_decay_epochs:
- 50
n_epochs_warmup: 1
make_pair_online: false
gradient_max_norm: 'Infinity'
max_step_per_epoch: 10000
n_rendering_workers: 1
save_epoch_interval: 100
n_dataloader_workers: 100
split_objects_across_gpus: true
ckpt_dir: /home/salman/rai_workspace/FoundationPose/learning/training/../../weights/2024-01-11-20-02-45/model_best.pth

[__init__()] self.h5_file:None
[__init__()] Using pretrained model from /home

Warp 1.13.0 initialized:
   CUDA Toolkit 12.9, Driver 12.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA RTX 6000 Ada Generation" (48 GiB, sm_89, mempool enabled)
   Kernel cache:
     /home/salman/.cache/warp/1.13.0


/home/salman/rai_workspace/FoundationPose/learning/training/predict_score.py:151: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_dir)
[__init__()] init

num original candidates = 252
num of pose after clustering: 252


In [ ]:
def sam3_text_detect(image_pil: Image.Image) -> list[dict]:
    """SAM3 text-prompt path. Returns list of {mask, score, box_xyxy}."""
    inputs = sam3_processor(
        images         = image_pil,
        text           = DETECTION_PROMPT,
        return_tensors = "pt",
    ).to(DEVICE)

    with torch.no_grad():
        outputs = sam3_model(**inputs)

    results = sam3_processor.post_process_instance_segmentation(
        outputs,
        threshold      = SAM3_SCORE_THRESH,
        mask_threshold = SAM3_MASK_THRESH,
        target_sizes   = inputs.get("original_sizes").tolist(),
    )[0]

    detections = []
    for mask_t, score_t, box_t in zip(
        results["masks"], results["scores"], results["boxes"]
    ):
        mask = mask_t.cpu().numpy().astype(bool)
        if not mask.any():
            continue
        detections.append({
            "mask":     mask,
            "score":    float(score_t),
            "box_xyxy": box_t.cpu().numpy(),
            "source":   "sam3_text",
        })
    return detections


def gdino_sam3_box_detect(image_pil: Image.Image, W: int, H: int) -> list[dict]:
    """GroundingDINO bbox → SAM3 box-prompt fallback. Returns list of {mask, score, box_xyxy}."""
    import tempfile, os
    from groundingdino.util.inference import load_image, predict

    model = load_gdino()

    # GroundingDINO requires a file path
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        tmp_path = tmp.name
    image_pil.save(tmp_path)

    try:
        _, img_tensor = load_image(tmp_path)
        boxes_norm, logits, _ = predict(
            model          = model,
            image          = img_tensor,
            caption        = DETECTION_PROMPT,
            box_threshold  = GD_BOX_THRESHOLD,
            text_threshold = GD_TEXT_THRESHOLD,
        )
    finally:
        os.unlink(tmp_path)

    if len(logits) == 0:
        return []

    boxes_xyxy = boxes_cxcywh_to_xyxy(boxes_norm, W, H)
    order      = logits.argsort(descending=True)[:MAX_PER_CAMERA]
    detections = []

    for i in order:
        score = logits[i].item()
        box   = boxes_xyxy[i].numpy()

        inputs = sam3_processor(
            images             = image_pil,
            input_boxes        = [[box.tolist()]],
            input_boxes_labels = [[1]],
            return_tensors     = "pt",
        ).to(DEVICE)

        with torch.no_grad():
            outputs = sam3_model(**inputs)

        results = sam3_processor.post_process_instance_segmentation(
            outputs,
            threshold      = 0.0,          # box mode: accept whatever SAM3 returns
            mask_threshold = SAM3_MASK_THRESH,
            target_sizes   = inputs.get("original_sizes").tolist(),
        )[0]

        if len(results["masks"]) == 0:
            continue

        mask = results["masks"][0].cpu().numpy().astype(bool)
        if not mask.any():
            continue

        detections.append({
            "mask":     mask,
            "score":    score, 
            "box_xyxy": box,
            "source":   "gdino_sam3_box",
        })

    return detections


with open(CAM_INFO) as f:
    camera_info = json.load(f)

print(f"\nDetecting: '{DETECTION_PROMPT}'")
all_detections: list[dict] = []

for cam_name, info in camera_info.items():
    W, H      = info["width"], info["height"]
    image_pil = Image.open(info["rgb_path"]).convert("RGB")

    dets = sam3_text_detect(image_pil)

    if dets:
        print(f"  {cam_name}: SAM3 text → {len(dets)} detection(s)")
    else:
        print(f"  {cam_name}: SAM3 text found nothing — trying GroundingDINO fallback …")
        dets = gdino_sam3_box_detect(image_pil, W, H)
        if dets:
            print(f"  {cam_name}: GD fallback → {len(dets)} detection(s)")
        else:
            print(f"  {cam_name}: no detection")
            continue

    dets.sort(key=lambda d: d["score"], reverse=True)
    for det in dets[:MAX_PER_CAMERA]:
        print(f"    score={det['score']:.3f}  source={det['source']}")
        det["cam"]  = cam_name
        det["info"] = info
        all_detections.append(det)

if not all_detections:
    raise RuntimeError(
        "No detections on any camera. Try lowering SAM3_SCORE_THRESH or GD_BOX_THRESHOLD."
    )

all_detections.sort(key=lambda d: d["score"], reverse=True)
top_dets = all_detections[:MAX_TOTAL]
print(f"\n{len(top_dets)} detection(s) selected for FoundationPose")


Detecting: 'yellow lamp'
  cam_dim_0: SAM3 text → 2 detection(s)
    score=0.901  source=sam3_text
    score=0.108  source=sam3_text
  cam_dim_1: SAM3 text → 1 detection(s)
    score=0.613  source=sam3_text
  cam_dim_2: SAM3 text → 2 detection(s)
    score=0.412  source=sam3_text
    score=0.107  source=sam3_text
  cam_dim_3: SAM3 text → 3 detection(s)
    score=0.137  source=sam3_text
    score=0.121  source=sam3_text
  cam_dim_4: SAM3 text → 3 detection(s)
    score=0.573  source=sam3_text
    score=0.315  source=sam3_text

8 detection(s) selected for FoundationPose


In [ ]:
sam3_model.to("cpu")
torch.cuda.empty_cache()

print("\nRunning FoundationPose …")
estimates: list[dict] = []

for idx, det in enumerate(top_dets):
    info = det["info"]
    print(f"  [{idx+1}/{len(top_dets)}]  cam={det['cam']}  score={det['score']:.3f}  source={det['source']}")

    rgb   = np.array(Image.open(info["rgb_path"]).convert("RGB"))
    depth = np.load(info["depth_path"]).astype(np.float32)
    K     = np.array(info["K"],           dtype=np.float64)
    T_wc  = np.array(info["T_world_cam"], dtype=np.float64)

    try:
        T_co = fp_estimator.register(
            K         = K,
            rgb       = rgb,
            depth     = depth,
            ob_mask   = det["mask"],
            iteration = N_FP_ITERATIONS,
        )
    except Exception as e:
        print(f"    FoundationPose failed: {e}")
        continue

    T_wo      = T_wc @ T_co
    pos       = T_wo[:3, 3]
    quat_xyzw = Rotation.from_matrix(T_wo[:3, :3]).as_quat()
    quat_wxyz = np.array([quat_xyzw[3], *quat_xyzw[:3]])   # RAI [w,x,y,z]

    print(f"    → world pos = {np.round(pos, 4)}")
    estimates.append({
        "position":    pos,
        "quat_wxyz":   quat_wxyz,
        "T_world_obj": T_wo,
        "T_cam_obj":   T_co,
        "score":       det["score"],
        "cam":         det["cam"],
        "box_xyxy":    det["box_xyxy"],
        "mask":        det["mask"],
        "source":      det["source"],
        "info":        info,
    })

if not estimates:
    raise RuntimeError("FoundationPose produced no estimates. Check GPU memory and logs above.")

print(f"\n{len(estimates)} pose estimate(s) obtained")

[register()] Welcome
[register()] poses:(252, 4, 4)
[register()] after viewpoint, add_errs min:-1.0
/home/salman/miniconda3/envs/robotics_proj_env/lib/python3.10/site-packages/torch/__init__.py:1144: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)
[predict()] ob_in_cams:(252, 4, 4)
[predict()] self.cfg.use_normal:False
[predict()] trans_normalizer:[0.019999999552965164, 0.019999999552965164, 0.05000000074505806], rot_normalizer:0.3490658503988659



Running FoundationPose …
  [1/8]  cam=cam_dim_0  score=0.901  source=sam3_text
Module Utils 9293bed load on device 'cuda:0' took 0.36 ms  (cached)


[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
/home/salman/rai_workspace/FoundationPose/learning/training/predict_pose_refine.py:190: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.amp):
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[m

    → world pos = [-0.621  -0.3422  0.6706]
  [2/8]  cam=cam_dim_1  score=0.613  source=sam3_text


[predict()] ob_in_cams:(252, 4, 4)
[predict()] self.cfg.use_normal:False
[predict()] trans_normalizer:[0.019999999552965164, 0.019999999552965164, 0.05000000074505806], rot_normalizer:0.3490658503988659
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()

...

    → world pos = [-0.6201 -0.3422  0.6724]
  [7/8]  cam=cam_dim_3  score=0.121  source=sam3_text


[predict()] ob_in_cams:(252, 4, 4)
[predict()] self.cfg.use_normal:False
[predict()] trans_normalizer:[0.019999999552965164, 0.019999999552965164, 0.05000000074505806], rot_normalizer:0.3490658503988659
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()

    → world pos = [-0.6201 -0.3427  0.6715]
  [8/8]  cam=cam_dim_0  score=0.108  source=sam3_text


[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_cr

    → world pos = [-0.6202 -0.342   0.6708]

8 pose estimate(s) obtained


In [8]:
clusters: list[dict] = []
for est in estimates:
    placed = False
    for cl in clusters:
        if np.linalg.norm(est["position"] - cl["rep"]["position"]) < CLUSTER_DIST_M:
            cl["members"].append(est)
            if est["score"] > cl["rep"]["score"]:
                cl["rep"] = est
            placed = True
            break
    if not placed:
        clusters.append({"rep": est, "members": [est]})

clusters.sort(key=lambda c: c["rep"]["score"], reverse=True)
print(f"{len(clusters)} unique object(s) found after clustering\n")

1 unique object(s) found after clustering



In [9]:
print("=" * 70)
print("POSE ESTIMATION RESULTS  (FoundationPose + SAM3)")
print("=" * 70)

for i, cl in enumerate(clusters):
    rep          = cl["rep"]
    pos          = rep["position"]
    nearest_name = min(GT, key=lambda n: np.linalg.norm(pos - GT[n][:3]))
    pos_err      = np.linalg.norm(pos - GT[nearest_name][:3])

    print(f"\nObject {i+1}  →  likely '{nearest_name}'")
    print(f"  Estimated position   [x,y,z]   : {np.round(pos, 4)}")
    print(f"  Estimated quaternion [w,x,y,z] : {np.round(rep['quat_wxyz'], 4)}")
    print(f"  Best detection score           : {rep['score']:.3f}")
    print(f"  Best camera                    : {rep['cam']}")
    print(f"  Detection source               : {rep['source']}")
    print(f"  Observations merged            : {len(cl['members'])}")
    print(f"  Position error vs GT           : {pos_err*100:.1f} cm")
    print(f"  4×4 T_world_obj:\n{rep['T_world_obj']}")

print("\n" + "=" * 70)
print("GROUND TRUTH (from mesh_clutter_high_30.g)")
print("=" * 70)
for name, gt in GT.items():
    print(f"  {name}:")
    print(f"    position   [x,y,z]   = {gt[:3]}")
    print(f"    quaternion [w,x,y,z] = {gt[3:]}")

POSE ESTIMATION RESULTS  (FoundationPose + SAM3)

Object 1  →  likely 'goal_yellow_lamp_9_mesh'
  Estimated position   [x,y,z]   : [-0.621  -0.3422  0.6706]
  Estimated quaternion [w,x,y,z] : [ 0.2343 -0.3771  0.7603  0.4742]
  Best detection score           : 0.901
  Best camera                    : cam_dim_0
  Detection source               : sam3_text
  Observations merged            : 8
  Position error vs GT           : 0.1 cm
  4×4 T_world_obj:
[[-0.60580689 -0.79561049 -0.0013632  -0.62096423]
 [-0.35118067  0.26586354  0.89776868 -0.34222308]
 [-0.71391189  0.54435313 -0.44046506  0.6706233 ]
 [ 0.          0.          0.          1.        ]]

GROUND TRUTH (from mesh_clutter_high_30.g)
  goal_yellow_lamp_9_mesh:
    position   [x,y,z]   = [-0.620966 -0.342511  0.671156]
    quaternion [w,x,y,z] = [ 0.287543 -0.46152   0.714029  0.441   ]
  goal_pose_yellow_lamp_12_mesh:
    position   [x,y,z]   = [0.219547 0.280159 0.698313]
    quaternion [w,x,y,z] = [ 0.38692   0.        0. 

In [ ]:
print("\nGenerating visualisations …")
mesh_obj  = trimesh.load(str(MESH_PATH))
verts     = np.array(mesh_obj.vertices)
verts_hom = np.hstack([verts, np.ones((len(verts), 1))])

PALETTE = [
    (0,   255,   0),    # green   — object 1
    (0,   165, 255),    # orange  — object 2
    (255,   0, 255),    # magenta
    (255, 255,   0),    # cyan
]

for i, cl in enumerate(clusters):
    rep      = cl["rep"]
    color    = PALETTE[i % len(PALETTE)]
    vis      = cv2.imread(rep["info"]["rgb_path"])
    H_v, W_v = vis.shape[:2]
    K_v      = np.array(rep["info"]["K"])
    T_co     = rep["T_cam_obj"]

    # semi-transparent mask overlay
    mask_u8 = rep["mask"].astype(np.uint8) * 255
    colored = np.zeros_like(vis)
    colored[:] = color[::-1]   # BGR
    vis = cv2.addWeighted(
        vis, 1.0,
        cv2.bitwise_and(colored, colored, mask=mask_u8), 0.4,
        0,
    )

    x1, y1, x2, y2 = rep["box_xyxy"].astype(int)
    cv2.rectangle(vis, (x1, y1), (x2, y2), color, 4)

    # label — include detection source
    nearest = min(GT, key=lambda n: np.linalg.norm(rep["position"] - GT[n][:3]))
    short   = "lamp" if "9_mesh" in nearest else "target"
    label   = f"{short} [{rep['source']}] s={rep['score']:.2f}"
    cv2.putText(vis, label, (x1, max(y1 - 15, 30)), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3)

    axis_len = 0.05
    o  = project_pt(T_co[:3, 3],                           K_v)
    xp = project_pt(T_co[:3, 3] + axis_len * T_co[:3, 0], K_v)
    yp = project_pt(T_co[:3, 3] + axis_len * T_co[:3, 1], K_v)
    zp = project_pt(T_co[:3, 3] + axis_len * T_co[:3, 2], K_v)
    cv2.arrowedLine(vis, o, xp, (0,   0, 255), 4, tipLength=0.3)
    cv2.arrowedLine(vis, o, yp, (0, 255,   0), 4, tipLength=0.3)
    cv2.arrowedLine(vis, o, zp, (255,   0,   0), 4, tipLength=0.3)

    verts_cam = (T_co @ verts_hom.T).T[:, :3]
    front = verts_cam[:, 2] > 0
    if front.any():
        pts      = (K_v @ verts_cam[front].T).T
        pts      = (pts[:, :2] / pts[:, 2:3]).astype(int)
        in_frame = ((pts[:, 0] >= 0) & (pts[:, 0] < W_v) &
                    (pts[:, 1] >= 0) & (pts[:, 1] < H_v))
        for pt in pts[in_frame][::8]:
            cv2.circle(vis, tuple(pt), 3, color, -1)

    pos = rep["position"]
    txt = f"pos=[{pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f}]"
    cv2.putText(vis, txt, (20, H_v - 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)
    cv2.putText(vis, txt, (20, H_v - 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 1)

    out_path = PROJECT_ROOT / "outputs" / f"pose_fp_sam3_object{i+1}_{short}.png"
    cv2.imwrite(str(out_path), vis)
    print(f"  Saved -> {out_path}")

print("\nDone.")


Generating visualisations …
  Saved -> /home/salman/rai_workspace/foundation_pose_section/outputs/pose_fp_sam3_object1_lamp.png

Done.
